# Runbook 1.5.b: AI analyzing code for vulnerabilities  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Detect Attacks, Collaborate · **Maps to:** 1.5.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Build the pattern-matching core of an AI code security scanner. Given a string of Python source
code, return a sorted list of vulnerability tags for every dangerous pattern found. This is the
rule-evaluation layer that tools like Bandit and Semgrep implement under the hood.

**Lesson objective.** Explain how AI-powered static analysis scans source code for known vulnerability patterns, identify specific risky constructs in Python code, and articulate why each flagged pattern creates a security risk.

### What comes in
A single string, the entire source code to scan. May be multi-line or empty.

### What must come out
A sorted `list[str]` of vulnerability tags drawn from:
- `"vuln:eval"`
- `"vuln:hardcoded_password"`
- `"vuln:md5"`
- `"vuln:os_system"`
- `"vuln:pickle_loads"`
- `"vuln:subprocess_shell"`

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
'result = eval(user_input)'
    -> ["vuln:eval"]

'os.system("ls " + path)\nsubprocess.run(cmd, shell=True)'
    -> ["vuln:os_system", "vuln:subprocess_shell"]

'import hashlib\nhashlib.md5(data)\npickle.loads(blob)'
    -> ["vuln:md5", "vuln:pickle_loads"]

'x = 1 + 2'
    -> []
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] `password =` matching is case-insensitive; all other patterns are case-sensitive.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(code: str) -> list[str]
```

#### Input (`code`)
A single string containing Python source code to scan. May be multi-line. May be empty.

#### Output (`result`)
A **sorted, de-duplicated** `list[str]` of vulnerability finding tags. Return `[]` if no
patterns are found.

#### Pattern rule set (case-sensitive substring search unless noted)

| Tag returned                   | Trigger substring         | Vulnerability class                      |
|--------------------------------|--------------------------|------------------------------------------|
| `"vuln:eval"`                  | `eval(`                  | Code injection via eval                  |
| `"vuln:os_system"`             | `os.system(`             | OS command injection                     |
| `"vuln:subprocess_shell"`      | `shell=True`             | OS command injection via subprocess      |
| `"vuln:md5"`                   | `md5(`                   | Broken cryptographic hash (MD5)          |
| `"vuln:pickle_loads"`          | `pickle.loads(`          | Unsafe deserialization                   |
| `"vuln:hardcoded_password"`    | `password =`             | Hardcoded credential in source           |

Notes:
- `password =` matching is case-insensitive (matches `Password =`, `PASSWORD =`, etc.).
- All other patterns are case-sensitive (Python identifiers are case-sensitive).
- A single line may trigger multiple tags if multiple patterns are present.
- Comments are NOT excluded, a dangerous call inside a comment still triggers the tag.
  (In practice, AST-based tools skip comments; here we use substring scan for transparency.)

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Output always sorted ascending (lexicographic) and de-duplicated.
- `solve("")` returns `[]`.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import List

RULES = [
    ("vuln:eval",                "eval(",        True),
    ("vuln:hardcoded_password",  "password =",   False),
    ("vuln:md5",                 "md5(",         True),
    ("vuln:os_system",           "os.system(",   True),
    ("vuln:pickle_loads",        "pickle.loads(", True),
    ("vuln:subprocess_shell",    "shell=True",   True),
]


def solve(code: str) -> List[str]:
    if not code:
        return []
    findings = set()
    for tag, trigger, case_sensitive in RULES:
        haystack = code if case_sensitive else code.lower()
        needle = trigger if case_sensitive else trigger.lower()
        if needle in haystack:
            findings.add(tag)
    return sorted(findings)


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

import os


def test_eval_only():
    assert impl.solve("result = eval(user_input)") == ["vuln:eval"]


def test_os_system_and_shell_true():
    code = 'os.system("ls " + path)\nsubprocess.run(cmd, shell=True)'
    assert impl.solve(code) == ["vuln:os_system", "vuln:subprocess_shell"]


def test_md5_and_pickle():
    code = "import hashlib\nhashlib.md5(data)\npickle.loads(blob)"
    assert impl.solve(code) == ["vuln:md5", "vuln:pickle_loads"]


def test_clean_code():
    assert impl.solve("x = 1 + 2\nprint(x)") == []


def test_empty_string():
    assert impl.solve("") == []


def test_hardcoded_password_lowercase():
    assert impl.solve('password = "secret123"') == ["vuln:hardcoded_password"]


def test_hardcoded_password_mixed_case():
    assert impl.solve('Password = "abc"') == ["vuln:hardcoded_password"]


def test_hardcoded_password_uppercase():
    assert impl.solve('PASSWORD = "abc"') == ["vuln:hardcoded_password"]


def test_all_patterns():
    code = (
        "eval(x)\n"
        "os.system(cmd)\n"
        "subprocess.run(args, shell=True)\n"
        "hashlib.md5(data)\n"
        "pickle.loads(blob)\n"
        'password = "pw"\n'
    )
    assert impl.solve(code) == [
        "vuln:eval",
        "vuln:hardcoded_password",
        "vuln:md5",
        "vuln:os_system",
        "vuln:pickle_loads",
        "vuln:subprocess_shell",
    ]


def test_dedup():
    code = "eval(a)\neval(b)\neval(c)"
    result = impl.solve(code)
    assert result == sorted(set(result))
    assert result == ["vuln:eval"]


def test_sorted_output():
    code = "pickle.loads(x)\neval(y)"
    result = impl.solve(code)
    assert result == sorted(result)


def test_no_false_positive_md5_variable():
    assert impl.solve("md5_value = compute_hash()") == []


def test_shell_false_not_flagged():
    assert impl.solve("subprocess.run(cmd, shell=False)") == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve('result = eval(user_input)'),
     ["vuln:eval"]),
    (lambda: solve('os.system("ls " + path)\nsubprocess.run(cmd, shell=True)'),
     ["vuln:os_system", "vuln:subprocess_shell"]),
    (lambda: solve('import hashlib\nhashlib.md5(data)\npickle.loads(blob)'),
     ["vuln:md5", "vuln:pickle_loads"]),
    (lambda: solve('x = 1 + 2'),
     []),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it